# Chest X-Ray Multi-Label Disease Classification — 4-Model Architecture Comparison

**ICT 4442 — Deep Learning Mini Project**

Compares four architectures spanning distinct families on the CheXpert dataset:
- **MLP** (classical baseline, image-only)
- **ResNet18** (CNN)
- **CNN-LSTM** (RNN/LSTM — patch-sequence)
- **ViT-B/16** (attention-based)

All four are trained under an identical protocol: same train/val/test split, same
preprocessing, same loss (multi-label BCE), same evaluation metrics (per-disease
AUC-ROC, precision, recall, F1). This is a **multi-label** problem (5 independent
binary diseases per image), not multi-class — a patient can have multiple
conditions simultaneously, so BCEWithLogitsLoss + sigmoid is used throughout,
not CrossEntropyLoss + softmax.


---

**This notebook (`00_shared_pipeline.ipynb`) — Owner: Advait Gujar (230911404)**

Builds the shared data pipeline and evaluation infrastructure used identically by all four models: data loading/cleaning, the fixed 80/10/10 split, the Dataset class and transforms, class weights, the shared loss, and the shared `train_model`/`evaluate` functions. Each model’s own architecture and training run lives in its own notebook (`01_mlp_baseline.ipynb`, `02_resnet18.ipynb`, `03_cnnlstm.ipynb`, `04_vit.ipynb`), and the final 4-model comparison is in `05_results_comparison.ipynb`. Run this notebook first, in the same kernel the others will use.

## 1. Imports and Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights, vit_b_16, ViT_B_16_Weights
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    precision_recall_fscore_support, confusion_matrix
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


## 2. Configuration

In [2]:
DATASET_ROOT = r'D:\Datasets\CheXpert-v1.0-small'
OUTPUT_DIR   = r'D:\Projects\ChestXray-Academic-Comparison\outputs'
CHECKPOINT_DIR = r'D:\Projects\ChestXray-Academic-Comparison\checkpoints'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DISEASE_COLS = [
    'Atelectasis',
    'Cardiomegaly',
    'Consolidation',
    'Edema',
    'Pleural Effusion'
]

U_ONES  = ['Atelectasis', 'Edema']            # uncertain (-1) -> treated as positive (1)
U_ZEROS = ['Cardiomegaly', 'Consolidation', 'Pleural Effusion']  # uncertain (-1) -> treated as negative (0)

IMG_SIZE    = 224
BATCH_SIZE  = 16
NUM_WORKERS = 0
NUM_EPOCHS  = 5   # reduce for a quicker first pass; 15-20 for final results

print(f"Dataset root: {DATASET_ROOT}")
print(f"Diseases ({len(DISEASE_COLS)}): {DISEASE_COLS}")

Dataset root: D:\Datasets\CheXpert-v1.0-small
Diseases (5): ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']


## 3. Data Loading and Cleaning

CheXpert labels are `1` (positive), `0` (negative), `-1` (uncertain), or blank
(not mentioned). We apply the standard "U-Ones/U-Zeros" policy: uncertain labels
for diseases where uncertainty tends to reflect real subtle findings
(Atelectasis, Edema) are mapped to positive; uncertain labels for diseases where
it more often reflects genuine ambiguity (Cardiomegaly, Consolidation, Pleural
Effusion) are mapped to negative. Blank/NaN (not mentioned) is treated as negative.


In [3]:
def load_and_clean_csv(csv_path, dataset_root):
    df = pd.read_csv(csv_path)

    # Frontal-view images only (lateral views excluded for consistency)
    df = df[df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)

    # Fix image paths to point at the actual local dataset location
    df['Path'] = df['Path'].apply(
        lambda x: os.path.join(dataset_root, x.replace('CheXpert-v1.0-small/', ''))
    )

    # Not-mentioned (NaN) -> negative
    df[DISEASE_COLS] = df[DISEASE_COLS].fillna(0)

    # Uncertain (-1) label policy
    for col in U_ONES:
        df[col] = df[col].replace(-1, 1)
    for col in U_ZEROS:
        df[col] = df[col].replace(-1, 0)

    return df


csv_path = os.path.join(DATASET_ROOT, 'train.csv')
full_df = load_and_clean_csv(csv_path, DATASET_ROOT)
print(f"Total frontal-view images: {len(full_df)}")
full_df[DISEASE_COLS].sum()

Total frontal-view images: 191027


Atelectasis         59583.0
Cardiomegaly        23385.0
Consolidation       12983.0
Edema               61493.0
Pleural Effusion    76899.0
dtype: float64

## 4. Train / Validation / Test Split

CheXpert's official `valid.csv` is a tiny (~200-image) leaderboard-style set,
not suited for real model selection or a held-out test set. Instead, we split
`train.csv` itself into **train (80%) / validation (10%) / test (10%)**, fixed
with `random_state=42` and reused identically for all four models — this is
what makes the cross-model comparison valid.


In [4]:
# First split off the test set (10%), then split remaining 90% into train/val
train_val_df, test_df = train_test_split(
    full_df, test_size=0.10, random_state=SEED,
    stratify=full_df['Pleural Effusion'].astype(int)
)

train_df, val_df = train_test_split(
    train_val_df, test_size=(0.10 / 0.90), random_state=SEED,  # 10% of original total
    stratify=train_val_df['Pleural Effusion'].astype(int)
)

print(f"Train: {len(train_df)}  ({len(train_df)/len(full_df)*100:.1f}%)")
print(f"Val:   {len(val_df)}  ({len(val_df)/len(full_df)*100:.1f}%)")
print(f"Test:  {len(test_df)}  ({len(test_df)/len(full_df)*100:.1f}%)")

# Save the splits so every model (and every team member) uses the exact
# same rows, even if this notebook is re-run on a different machine.
train_df.to_csv(os.path.join(OUTPUT_DIR, 'train_split.csv'), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, 'val_split.csv'), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, 'test_split.csv'), index=False)

Train: 152821  (80.0%)
Val:   19103  (10.0%)
Test:  19103  (10.0%)


## 5. Dataset Class and Preprocessing Pipeline

In [5]:
def get_transforms(mode='train'):
    if mode == 'train':
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])


class CheXpertDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['Path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor(row[DISEASE_COLS].values.astype(np.float32), dtype=torch.float32)
        return img, labels


train_ds = CheXpertDataset(train_df, transform=get_transforms('train'))
val_ds   = CheXpertDataset(val_df,   transform=get_transforms('val'))
test_ds  = CheXpertDataset(test_df,  transform=get_transforms('val'))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("DataLoaders ready.")

DataLoaders ready.


## 6. Class Weights (for imbalanced multi-label BCE loss)

In [6]:
def get_class_weights(df, device):
    weights = []
    for col in DISEASE_COLS:
        pos = (df[col] == 1).sum()
        neg = (df[col] == 0).sum()
        weight = neg / (pos + 1e-6)
        weights.append(weight)
        print(f"{col:20s} pos={pos:6d}  neg={neg:6d}  weight={weight:.2f}")
    return torch.tensor(weights, dtype=torch.float32).to(device)

pos_weights = get_class_weights(train_df, device)

Atelectasis          pos= 47657  neg=105164  weight=2.21
Cardiomegaly         pos= 18759  neg=134062  weight=7.15
Consolidation        pos= 10436  neg=142385  weight=13.64
Edema                pos= 49292  neg=103529  weight=2.10
Pleural Effusion     pos= 61519  neg= 91302  weight=1.48


In [13]:
# Shared loss — identical BCEWithLogitsLoss (with per-disease class weights)
# is used by every model, so the comparison isolates architecture, not loss choice.
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
print('Shared criterion ready.')

Shared criterion ready.


## 7. Shared Training and Evaluation Functions

Multi-label setup throughout: `BCEWithLogitsLoss` (with class weights), sigmoid for probabilities, per-disease AUC-ROC as the primary metric. Used unchanged by all four model notebooks.

In [9]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0
    all_labels, all_preds = [], []

    for images, labels in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        if device.type == 'cuda':
            with torch.autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        all_labels.append(labels.cpu().detach().numpy())
        all_preds.append(torch.sigmoid(outputs).cpu().detach().numpy())

    all_labels = np.concatenate(all_labels)
    all_preds  = np.concatenate(all_preds)
    avg_loss   = total_loss / len(loader)
    aucs = _per_disease_auc(all_labels, all_preds)
    return avg_loss, float(np.mean(aucs))


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_labels, all_preds = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating', leave=False):
            images, labels = images.to(device), labels.to(device)
            if device.type == 'cuda':
                with torch.autocast(device_type='cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.item()
            all_labels.append(labels.cpu().numpy())
            all_preds.append(torch.sigmoid(outputs).cpu().numpy())

    all_labels = np.concatenate(all_labels)
    all_preds  = np.concatenate(all_preds)
    avg_loss   = total_loss / len(loader)
    aucs = _per_disease_auc(all_labels, all_preds)
    return avg_loss, float(np.mean(aucs)), all_labels, all_preds


def _per_disease_auc(labels, preds):
    aucs = []
    for i in range(labels.shape[1]):
        try:
            aucs.append(roc_auc_score(labels[:, i], preds[:, i]))
        except Exception:
            aucs.append(0.0)
    return aucs


def train_model(model, model_name, epochs=NUM_EPOCHS, lr=1e-4, freeze_schedule=False):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler(device='cuda') if device.type == 'cuda' else None

    history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}
    best_auc = 0.0

    for epoch in range(1, epochs + 1):
        if freeze_schedule and hasattr(model, 'backbone'):
            if epoch == 1:
                print(f"[{model_name}] Epochs 1-2: backbone frozen")
                for p in model.backbone.parameters():
                    p.requires_grad = False
            if epoch == 3:
                print(f"[{model_name}] Epoch 3+: full model unfrozen")
                for p in model.backbone.parameters():
                    p.requires_grad = True

        train_loss, train_auc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss, val_auc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_auc'].append(train_auc)
        history['val_auc'].append(val_auc)

        print(f"[{model_name}] Epoch {epoch}/{epochs}  Train Loss: {train_loss:.4f}  Train AUC: {train_auc:.4f}  "
              f"Val Loss: {val_loss:.4f}  Val AUC: {val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'{model_name}_best.pth'))

    # reload best checkpoint before returning
    model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, f'{model_name}_best.pth')))
    return model, history

---
**Next:** run `01_mlp_baseline.ipynb`, `02_resnet18.ipynb`, `03_cnnlstm.ipynb`, `04_vit.ipynb` (any order, same kernel), then `05_results_comparison.ipynb`.